In [ ]:
%matplotlib widget

import rospy
import actionlib
from assignment_2_2024.msg import PlanningAction, PlanningGoal
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from ipywidgets import widgets, Layout
from IPython.display import display
from nav_msgs.msg import Odometry
from sensor_msgs.msg import LaserScan

In [ ]:
class ActionClientManager:
    def __init__(self):
        self.client = actionlib.SimpleActionClient('/reaching_goal', PlanningAction)
        self.client.wait_for_server()
        self.reached_targets = 0
        self.not_reached_targets = 0
        self._lock = None  # Will link to shared lock if provided

    def set_lock(self, lock):
        self._lock = lock

    def send_goal(self, x, y, done_cb):
        goal = PlanningGoal()
        goal.target_pose.pose.position.x = x
        goal.target_pose.pose.position.y = y
        self.client.send_goal(goal, done_cb=lambda status, result: self._goal_done_cb(status, result, done_cb))

    def _goal_done_cb(self, status, result, user_cb):
        if self._lock:
            with self._lock:
                if status == actionlib.GoalStatus.SUCCEEDED:
                    self.reached_targets += 1
                else:
                    self.not_reached_targets += 1
        else:
            if status == actionlib.GoalStatus.SUCCEEDED:
                self.reached_targets += 1
            else:
                self.not_reached_targets += 1
        if user_cb:
            user_cb(status, result)

    def cancel_goal(self):
        self.client.cancel_goal()
        if self._lock:
            with self._lock:
                self.not_reached_targets += 1
        else:
            self.not_reached_targets += 1